In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("/kaggle/input/datasets/mnassrib/jena-climate/jena_climate_2009_2016.csv")
df.head(5)

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [3]:
column_mapping = {
    "Date Time": "date_time",
    "p (mbar)": "pressure",
    "T (degC)": "temperature",
    "Tpot (K)": "potential_temp",
    "Tdew (degC)": "dew_point",
    "rh (%)": "relative_humidity",
    "VPmax (mbar)": "max_vapor_pressure",
    "VPact (mbar)": "actual_vapor_pressure",
    "VPdef (mbar)": "vapor_pressure_deficit",
    "sh (g/kg)": "specific_humidity",
    "H2OC (mmol/mol)": "water_vapor_concentration",
    "rho (g/m**3)": "air_density",
    "wv (m/s)": "wind_speed",
    "max. wv (m/s)": "max_wind_speed",
    "wd (deg)": "wind_direction",
}

df.rename(columns=column_mapping, inplace=True)

with pd.option_context('display.max_columns',None):
    display(df.head())

,date_time,pressure,temperature,potential_temp,dew_point,relative_humidity,max_vapor_pressure,actual_vapor_pressure,vapor_pressure_deficit,specific_humidity,water_vapor_concentration,air_density,wind_speed,max_wind_speed,wind_direction
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [4]:
df['date_time'] = pd.to_datetime(df['date_time'], format='%d.%m.%Y %H:%M:%S')
df['date_time'].sample(5)

293439   2014-07-29 12:40:00
153882   2011-12-04 15:30:00
262013   2013-12-24 13:30:00
186043   2012-07-14 23:40:00
195504   2012-09-18 16:30:00
Name: date_time, dtype: datetime64[ns]

### data based on days only

In [5]:
df_daily = df.groupby(pd.Grouper(key='date_time', freq='D')).mean(numeric_only=True).reset_index()
print(df_daily.shape)
df_daily.head()

(2921, 15)


,date_time,pressure,temperature,potential_temp,dew_point,relative_humidity,max_vapor_pressure,actual_vapor_pressure,vapor_pressure_deficit,specific_humidity,water_vapor_concentration,air_density,wind_speed,max_wind_speed,wind_direction
0,2009-01-01,999.145594,-6.810629,266.414545,-8.015594,91.086014,3.691119,3.355524,0.335315,2.091049,3.357832,1305.178252,0.778601,1.378252,181.863077
1,2009-01-02,999.600625,-3.728194,269.463194,-4.824861,92.086806,4.640069,4.267292,0.373056,2.659792,4.268750,1290.353194,1.419514,2.227361,125.072014
2,2009-01-03,998.548611,-5.271736,268.002292,-9.015833,76.458056,4.184792,3.107708,1.077014,1.937778,3.111944,1297.117014,1.250903,2.065069,190.383333
3,2009-01-04,988.510694,-1.375208,272.685347,-2.897014,89.417361,5.524306,4.938958,0.584861,3.114028,4.997014,1264.634514,1.720417,3.564861,213.069861
4,2009-01-05,990.405694,-4.867153,269.039306,-6.797292,86.260417,4.362708,3.806736,0.555625,2.397014,3.847778,1284.372778,3.800278,5.940000,118.287361


In [6]:
df_daily.describe()

,date_time,pressure,temperature,potential_temp,dew_point,relative_humidity,max_vapor_pressure,actual_vapor_pressure,vapor_pressure_deficit,specific_humidity,water_vapor_concentration,air_density,wind_speed,max_wind_speed,wind_direction
count,2921,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000,2921.000000
mean,2012-12-31 01:05:04,989.220852,9.438993,283.480909,4.950669,76.031012,13.565876,9.530899,4.034893,6.020562,9.637279,1216.121931,1.700779,3.054484,174.772827
min,2009-01-01 00:00:00,948.995972,-16.457292,256.104028,-19.387222,37.580347,1.701597,1.312917,0.000000,0.810625,1.302014,1134.728472,-1248.217847,-1386.273264,0.000000
25%,2011-01-01 00:00:00,984.359861,3.808333,277.822986,0.363542,67.685764,8.096806,6.298333,1.339792,3.980625,6.383542,1188.101875,1.448542,2.494306,142.243611
50%,2012-12-31 00:00:00,989.530000,9.708333,283.850972,5.272500,76.988472,12.256597,8.937361,2.982986,5.624792,9.012292,1212.219167,1.903194,3.231806,188.648264
75%,2014-12-31 00:00:00,994.432917,15.444722,289.495556,9.955903,84.561319,17.953889,12.312014,5.693889,7.765069,12.424583,1241.004722,2.586806,4.253264,212.555000
max,2017-01-01 00:00:00,1013.957569,29.375347,303.124097,20.185486,100.000000,43.083403,23.749722,24.012778,15.101667,24.056806,1367.595417,7.644514,11.465208,279.445139
std,NaN,8.095683,7.840078,7.902083,6.583386,11.377820,6.878382,4.074847,3.548092,2.585796,4.123500,37.388088,23.155030,25.757632,54.411625


#### Scaling the data

In [7]:
from sklearn.preprocessing import StandardScaler

# 1. Feature columns (all numeric columns except date_time)
columns = [col for col in df_daily.columns if col != 'date_time']

# 2. Chronological split (80% train, 20% test)
split_idx = int(len(df_daily) * 0.8)

train_df = df_daily.iloc[:split_idx]
test_df  = df_daily.iloc[split_idx:]

# 3. Scaler for INPUT FEATURES (X)
scaler_x = StandardScaler()
train_x = scaler_x.fit_transform(train_df[columns])
test_x  = scaler_x.transform(test_df[columns])        # Use .transform() only!

# 4. Scaler for TARGET (y = temperature)
scaler_y = StandardScaler()
train_y = scaler_y.fit_transform(train_df[['temperature']])
test_y  = scaler_y.transform(test_df[['temperature']])      # Use .transform() only!


In [8]:
print(train_x.shape)
print(test_x.shape)
print(train_y.shape)
print(test_y.shape)

(2336, 14)
(585, 14)
(2336, 1)
(585, 1)


In [9]:
# getting the scaled y set 
scaler_y.inverse_transform(train_y)

array([[-6.81062937],
       [-3.72819444],
       [-5.27173611],
       ...,
       [13.53479167],
       [14.75534722],
       [13.81041667]])

#### Creating a 30 day sequence

The model would make prediction based on last 30 day data.

In [10]:
def create_sequence(data, target, length=30):
    x, y = [], []

    for i in range(len(data) - length):
        x.append(data[i : i + length])
        y.append(target[i + length])

    return np.array(x), np.array(y)


In [11]:
seq = 30

x_train_seq, y_train_seq = create_sequence(train_x, train_y, 30)
x_test_seq, y_test_seq = create_sequence(test_x, test_y, 30)

In [13]:
print(x_train_seq.shape)                       # each sample consists of 30 rows (30 days of data) and 14 features
print(x_test_seq.shape)                       # each sample consists of 30 rows (30 days of data) and 14 features
print()
print(y_train_seq.shape)
print()
print(x_train_seq[:3])

(2306, 30, 14)
(555, 30, 14)

(2306, 1)

[[[ 1.27738884 -2.01093478 -2.10348546 ... -1.43570534 -1.49015043
    0.13960093]
  [ 1.33375446 -1.61843041 -1.71808414 ... -0.76407655 -0.90902659
   -0.89404872]
  [ 1.20343922 -1.81497855 -1.90276721 ... -0.9407685  -1.02009775
    0.29467744]
  ...
  [ 0.49432268 -1.46125025 -1.49531793 ... -2.25162191 -2.43341518
   -3.17047454]
  [ 0.99315762 -1.44561625 -1.51954787 ... -0.58643856 -0.51735464
   -2.34579208]
  [ 1.24875579 -1.44879964 -1.5429877  ... -0.03955223  0.12640178
   -1.99499967]]

 [[ 1.33375446 -1.61843041 -1.71808414 ... -0.76407655 -0.90902659
   -0.89404872]
  [ 1.20343922 -1.81497855 -1.90276721 ... -0.9407685  -1.02009775
    0.29467744]
  [-0.03997926 -1.31881099 -1.31074898 ... -0.44875274  0.00634798
    0.70759313]
  ...
  [ 0.99315762 -1.44561625 -1.51954787 ... -0.58643856 -0.51735464
   -2.34579208]
  [ 1.24875579 -1.44879964 -1.5429877  ... -0.03955223  0.12640178
   -1.99499967]
  [ 0.68651379 -1.39858158 -1.44

### Sequential Model Training

In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential

In [15]:
model = keras.Sequential([
    layers.LSTM(64, return_sequences=False, input_shape=(30, train_x.shape[1])),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer = keras.optimizers.Adam(),
    loss = keras.losses.MeanSquaredError(),
    metrics = ['mae']
)

history = model.fit(
    x_train_seq, y_train_seq,
    validation_data=(x_test_seq, y_test_seq),
    epochs=15,
    batch_size=32
)

I0000 00:00:1791557829.786371      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13484 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791557829.790018      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.2448 - mae: 0.3697 - val_loss: 0.1341 - val_mae: 0.2974
Epoch 2/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1127 - mae: 0.2664 - val_loss: 0.1191 - val_mae: 0.2761
Epoch 3/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0978 - mae: 0.2464 - val_loss: 0.1022 - val_mae: 0.2601
Epoch 4/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0887 - mae: 0.2342 - val_loss: 0.0979 - val_mae: 0.2521
Epoch 5/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0823 - mae: 0.2234 - val_loss: 0.0900 - val_mae: 0.2391
Epoch 6/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0792 - mae: 0.2193 - val_loss: 0.1059 - val_mae: 0.2609
Epoch 7/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0749 - mae: 0.2129 - val_loss: 0.0923 - val_mae: 0.2413
Epoch 8/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0731 - mae: 0.2091 - val_loss: 0.0881 - val_mae: 0.2370
Epoch 9/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0703 - mae:

#### Making predictions and evaluations

In [16]:
test_loss, test_mae = model.evaluate(x_test_seq, y_test_seq)

print(test_loss)
print(test_mae)

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0846 - mae: 0.2292
0.08455122262239456
0.22917520999908447


In [17]:
y_pred = model.predict(x_test_seq)

# since y_pred is in scaled format, so we need to unscale it
y_pred_unscaled = scaler_y.inverse_transform(y_pred)
y_test_unscaled = scaler_y.inverse_transform(y_test_seq)

print("\nFirst 5 Days - Predicted vs Actual Temperature:")

for i in range(5):
    print(f"Day {i+1}: Predicted = {y_pred_unscaled[i][0]:.2f} °C  |  Actual = {y_test_unscaled[i][0]:.2f} °C")


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

First 5 Days - Predicted vs Actual Temperature:
Day 1: Predicted = 16.31 °C  |  Actual = 16.56 °C
Day 2: Predicted = 18.74 °C  |  Actual = 18.33 °C
Day 3: Predicted = 18.03 °C  |  Actual = 17.35 °C
Day 4: Predicted = 16.60 °C  |  Actual = 17.51 °C
Day 5: Predicted = 17.56 °C  |  Actual = 20.56 °C


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('MEAN ABSOLUTE ERROR', mean_absolute_error(y_test_unscaled, y_pred_unscaled))
print('MEAN SQUARED ERROR', mean_squared_error(y_test_unscaled, y_pred_unscaled))
print('R2 SCORE', r2_score(y_test_unscaled, y_pred_unscaled))

MEAN ABSOLUTE ERROR 1.7997701254841774
MEAN SQUARED ERROR 5.214572970476172
R2 SCORE 0.9098397108870152


---

#### Functional Model Training

In [20]:
inputs = keras.Input(shape=(30, 14))
x = layers.LSTM(64, return_sequences=True)(inputs)
x = layers.LSTM(128, return_sequences=True)(inputs)
x = layers.LSTM(346, return_sequences=True)(x)
x = layers.LSTM(64, return_sequences=False)(x)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=outputs)


model.compile(
    optimizer = keras.optimizers.Adam(0.001),
    loss = keras.losses.MeanSquaredError(),
    metrics = ['mae']
)

history = model.fit(
    x_train_seq, y_train_seq,
    validation_data = (x_test_seq, y_test_seq),
    epochs = 15,
    batch_size = 32,
    verbose = 1
)


Epoch 1/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.2393 - mae: 0.3857 - val_loss: 0.2367 - val_mae: 0.3970
Epoch 2/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1674 - mae: 0.3268 - val_loss: 0.1376 - val_mae: 0.3025
Epoch 3/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1150 - mae: 0.2671 - val_loss: 0.1135 - val_mae: 0.2600
Epoch 4/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0897 - mae: 0.2324 - val_loss: 0.0809 - val_mae: 0.2271
Epoch 5/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0773 - mae: 0.2176 - val_loss: 0.0778 - val_mae: 0.2240
Epoch 6/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0737 - mae: 0.2111 - val_loss: 0.0808 - val_mae: 0.2244
Epoch 7/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0715 - mae: 0.2077 - val_loss: 0.0793 - val_mae: 0.2229
Epoch 8/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0720 - mae: 0.2086 - val_loss: 0.0805 - val_mae: 0.2233
Epoch 9/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.072

In [21]:
test_loss, test_mae = model.evaluate(x_test_seq, y_test_seq)

print(test_loss)
print(test_mae)

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0976 - mae: 0.2469
0.0975584164261818
0.24686001241207123


In [22]:
y_pred = model.predict(x_test_seq)

# since y_pred is in scaled format, so we need to unscale it
y_pred_unscaled = scaler_y.inverse_transform(y_pred)
y_test_unscaled = scaler_y.inverse_transform(y_test_seq)

print("\nFirst 5 Days - Predicted vs Actual Temperature:")

for i in range(5):
    print(f"Day {i+1}: Predicted = {y_pred_unscaled[i][0]:.2f} °C  |  Actual = {y_test_unscaled[i][0]:.2f} °C")


18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step

First 5 Days - Predicted vs Actual Temperature:
Day 1: Predicted = 16.55 °C  |  Actual = 16.56 °C
Day 2: Predicted = 20.55 °C  |  Actual = 18.33 °C
Day 3: Predicted = 19.08 °C  |  Actual = 17.35 °C
Day 4: Predicted = 17.42 °C  |  Actual = 17.51 °C
Day 5: Predicted = 18.83 °C  |  Actual = 20.56 °C


In [23]:
print('MEAN ABSOLUTE ERROR', mean_absolute_error(y_test_unscaled, y_pred_unscaled))
print('MEAN SQUARED ERROR', mean_squared_error(y_test_unscaled, y_pred_unscaled))
print('R2 SCORE', r2_score(y_test_unscaled, y_pred_unscaled))

MEAN ABSOLUTE ERROR 1.9386534612124837
MEAN SQUARED ERROR 6.016771161145048
R2 SCORE 0.8959696545648357
